In [2]:
# 🧩 Nov 6 — Video Logging + System Optimization (Colab)
# ==========================================================
# - Reads video(s) from folder
# - Runs YOLOv8 PPE detection per frame (with resize + frame skip)
# - Associates PPE to nearest Person per frame (IoU/containment)
# - Triggers violations per-person ("No Helmet", "No Vest", "No Mask")
# - Saves evidence: annotated frames + clips (5s pre + 5s post)
# - Writes metadata (CSV + JSON) per saved artifact
# - Basic multi-threaded capture/processing with queues
# ==========================================================

# 1) Install dependencies
!pip install -q ultralytics opencv-python-headless==4.10.0.84 numpy torch pandas pyyaml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 66.4 MB/s eta 0:00:00


In [3]:
# 2) (Optional) Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
# 3) Imports & Paths
import os, cv2, time, json, math, threading, queue, uuid, shutil
import numpy as np
import pandas as pd
from collections import defaultdict, deque
from datetime import datetime
from ultralytics import YOLO
import torch

In [6]:
# 4) Project Paths (EDIT AS NEEDED)
# ================================
BASE_DIR = "/content/drive/MyDrive/SafetyEye"
MODEL_PATH = f"{BASE_DIR}/models/best_model/best.pt"      # <-- your trained weights
VIDEO_DIR  = f"{BASE_DIR}/realtime_inference/data"        # input videos
OUTPUT_DIR = f"{BASE_DIR}/violations"                     # evidence root
REPORT_DIR = f"{BASE_DIR}/violations_report"              # logs/metrics

os.makedirs(VIDEO_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("MODEL_PATH:", MODEL_PATH)
print("VIDEO_DIR:", VIDEO_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

BASE_DIR: /content/drive/MyDrive/SafetyEye
MODEL_PATH: /content/drive/MyDrive/SafetyEye/models/best_model/best.pt
VIDEO_DIR: /content/drive/MyDrive/SafetyEye/realtime_inference/data
OUTPUT_DIR: /content/drive/MyDrive/SafetyEye/violations


In [7]:
# 5) Inference & Optimization Config
# ================================
DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"
CONF_THRES    = 0.45     # detector confidence
IOU_THRES     = 0.5      # NMS IoU / association tolerance
IMG_SIZE      = 640
USE_FP16      = (DEVICE == "cuda")  # try fp16 on GPU
FRAME_RESIZE  = (1280, 720)  # downscale to speed up (w,h), set None to keep original
FRAME_SKIP    = 1        # process every Nth frame (1 = every frame; 2 = every 2nd, ...)
PRE_SEC       = 5.0      # seconds to keep before violation trigger (for clips)
POST_SEC      = 5.0      # seconds to keep after violation trigger

# ================================
# 6) Dataset classes (as per your data.yaml)
# ================================
CLASS_NAMES = [
    'Hardhat', 'Mask', 'NO-Hardhat', 'NO-Mask', 'NO-Safety Vest',
    'Person', 'Safety Cone', 'Safety Vest', 'machinery', 'vehicle'
]

IDX = {n:i for i,n in enumerate(CLASS_NAMES)}
CLS_PERSON       = IDX['Person']
CLS_HARDHAT      = IDX['Hardhat']
CLS_NO_HARDHAT   = IDX['NO-Hardhat']
CLS_VEST         = IDX['Safety Vest']
CLS_NO_VEST      = IDX['NO-Safety Vest']
CLS_MASK         = IDX['Mask']
CLS_NO_MASK      = IDX['NO-Mask']

# PPE association colors
GREEN = (0,255,0)
RED   = (0,0,255)
WHITE = (255,255,255)

In [8]:
# 7) Utilities
# ================================
def ts_now():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def ts_for_name():
    return datetime.now().strftime("%Y%m%d_%H%M%S")

def iou_xyxy(a, b):
    """IoU for [x1,y1,x2,y2] boxes"""
    xA = max(a[0], b[0]); yA = max(a[1], b[1])
    xB = min(a[2], b[2]); yB = min(a[3], b[3])
    inter = max(0, xB-xA) * max(0, yB-yA)
    if inter <= 0:
        return 0.0
    areaA = max(0,a[2]-a[0])*max(0,a[3]-a[1])
    areaB = max(0,b[2]-b[0])*max(0,b[3]-b[1])
    return inter / (areaA + areaB - inter + 1e-9)

def contains(inner, outer):
    """Return True if 'inner' box center lies in 'outer' box."""
    cx = 0.5*(inner[0]+inner[2]); cy = 0.5*(inner[1]+inner[3])
    return (cx >= outer[0] and cx <= outer[2] and cy >= outer[1] and cy <= outer[3])

def ensure_dir(p):
    os.makedirs(p, exist_ok=True)

# ================================
# 8) Load model
# ================================
model = YOLO(MODEL_PATH)
model.to(DEVICE)
print("✅ Model loaded on", DEVICE)

✅ Model loaded on cuda


In [9]:
# ================================
# 9) Lightweight person tracker (stable IDs across frames)
# ================================
class SimpleTracker:
    def __init__(self, iou_threshold=0.4, max_miss=30):
        self.iou_th = iou_threshold
        self.max_miss = max_miss
        self.next_id = 0
        self.tracks = {}   # id -> {'box': np.array([x1,y1,x2,y2]), 'miss': int}

    def _match(self, boxes):
        """Greedy IOU assignment: returns list of track_ids aligned with input boxes."""
        track_ids = [-1] * len(boxes)
        if not self.tracks or len(boxes) == 0:
            return track_ids

        # Build iou matrix (tracks x boxes)
        track_ids_list = list(self.tracks.keys())
        T = len(track_ids_list)
        B = len(boxes)
        ious = np.zeros((T, B), dtype=np.float32)
        for ti, tid in enumerate(track_ids_list):
            tb = self.tracks[tid]['box']
            for bi, bb in enumerate(boxes):
                ious[ti, bi] = iou_xyxy(tb, bb)

        used_tracks = set()
        used_boxes = set()
        # Greedy pair by IOU
        while True:
            ti, bi = np.unravel_index(np.argmax(ious), ious.shape)
            if ious[ti, bi] < self.iou_th:
                break
            if ti in used_tracks or bi in used_boxes:
                ious[ti, bi] = -1.0
                continue
            tid = track_ids_list[ti]
            track_ids[bi] = tid
            used_tracks.add(ti)
            used_boxes.add(bi)
            ious[ti, :] = -1.0
            ious[:, bi] = -1.0

        return track_ids

    def update(self, person_boxes):
        """Update tracks with current person boxes. Returns track_ids aligned with person_boxes."""
        person_boxes = [np.array(b, dtype=np.float32) for b in person_boxes]
        # match existing
        assign = self._match(person_boxes)

        # update matched tracks
        for bi, tid in enumerate(assign):
            if tid != -1:
                self.tracks[tid]['box'] = person_boxes[bi].copy()
                self.tracks[tid]['miss'] = 0

        # create new tracks for unmatched boxes
        for bi, tid in enumerate(assign):
            if tid == -1:
                tid = self.next_id
                self.next_id += 1
                self.tracks[tid] = {'box': person_boxes[bi].copy(), 'miss': 0}
                assign[bi] = tid

        # age unmatched tracks
        to_del = []
        for tid in self.tracks:
            if tid not in assign:
                self.tracks[tid]['miss'] += 1
                if self.tracks[tid]['miss'] > self.max_miss:
                    to_del.append(tid)
        for tid in to_del:
            self.tracks.pop(tid, None)

        return assign


# ================================
# 10) Per-person PPE association & rule evaluation (unchanged)
# ================================
def associate_ppe_to_persons(boxes, clss, confs, frame_w, frame_h):
    persons = []
    ppe_items = []
    for b, c, s in zip(boxes, clss, confs):
        c = int(c)
        if c == CLS_PERSON:
            persons.append({'box': b, 'score': float(s)})
        else:
            ppe_items.append({'box': b, 'cls': c, 'score': float(s)})

    results = []
    for pid, p in enumerate(persons):
        pbox = p['box']
        have_helmet = have_vest = have_mask = False
        neg_helmet = neg_vest = neg_mask = False
        conf_accum = []

        for item in ppe_items:
            b = item['box']; c = item['cls']; s = item['score']
            if contains(b, pbox) or iou_xyxy(b, pbox) > 0.1:
                conf_accum.append(s)
                if c == CLS_HARDHAT:      have_helmet = True
                elif c == CLS_VEST:       have_vest   = True
                elif c == CLS_MASK:       have_mask   = True
                elif c == CLS_NO_HARDHAT: neg_helmet  = True
                elif c == CLS_NO_VEST:    neg_vest    = True
                elif c == CLS_NO_MASK:    neg_mask    = True

        violations = []
        if neg_helmet or not have_helmet: violations.append("No Helmet")
        if neg_vest   or not have_vest:   violations.append("No Vest")
        if neg_mask   or not have_mask:   violations.append("No Mask")

        results.append({
            'pid': pid,
            'person_box': pbox.astype(int),
            'ppe': {'helmet': have_helmet, 'vest': have_vest, 'mask': have_mask},
            'violations': violations,
            'score': float(np.mean(conf_accum) if conf_accum else p['score'])
        })
    return results


def draw_person_overlays(frame, per_person):
    for info in per_person:
        x1,y1,x2,y2 = info['person_box']
        ok = (len(info['violations']) == 0)
        color = GREEN if ok else RED

        cv2.rectangle(frame, (x1,y1), (x2,y2), color, 2)
        badge_x = x1 + 4; badge_y = y1 - 8; step = 18
        def badge(txt, good, y):
            cv2.putText(frame, f"{txt}", (badge_x, max(12,y)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, GREEN if good else RED, 2)
        badge("Helmet", info['ppe']['helmet'], badge_y); badge_y += step
        badge("Vest",   info['ppe']['vest'],   badge_y); badge_y += step
        badge("Mask",   info['ppe']['mask'],   badge_y)

        if info.get('tid') is not None:
            cv2.putText(frame, f"ID:{info['tid']}", (x1, y2 + 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, WHITE, 2)

        if info['violations']:
            vtxt = " | ".join(info['violations'])
            cv2.putText(frame, f"⚠ {vtxt}", (x1, y2 + 20),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, RED, 2)
    return frame

In [10]:


# ================================
# 11) Processing Pipeline (threaded capture -> process)
# ================================
def process_video(path):
    video_name = os.path.basename(path)
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        print("❌ Cannot open:", path); return

    fps_in = cap.get(cv2.CAP_PROP_FPS) or 25.0
    writer = None
    out_path = os.path.join(OUTPUT_DIR, f"{os.path.splitext(video_name)[0]}_annotated.mp4")

    # ring buffer for pre-violation frames
    rb = RingBuffer(capacity=int(PRE_SEC * fps_in))
    ev = EvidenceLogger(OUTPUT_DIR, fps=fps_in, pre_sec=PRE_SEC, post_sec=POST_SEC)

    frame_idx = 0
    post_collect = {}  # pid -> remaining frames to keep after trigger

    t0_all = time.time()
    frame_times = []

    while True:
        ok, frame = cap.read()
        if not ok:
            break

        # resize for speed
        if FRAME_RESIZE is not None:
            frame = cv2.resize(frame, FRAME_RESIZE)

        # ring buffer always push original frame (annotated later)
        rb.push(frame, ts_now())

        # skip frames for speed
        if (frame_idx % FRAME_SKIP) != 0:
            # still write original to output if writer already initialized
            if writer is not None:
                writer.write(frame)
            frame_idx += 1
            continue

        h, w = frame.shape[:2]
        start = time.time()
        preds = model.predict(
            frame,
            conf=CONF_THRES,
            iou=IOU_THRES,
            imgsz=IMG_SIZE,
            device=DEVICE,
            half=USE_FP16,
            verbose=False
        )
        pred = preds[0]
        boxes = pred.boxes.xyxy.detach().cpu().numpy() if pred.boxes is not None else np.zeros((0,4))
        clss  = pred.boxes.cls.detach().cpu().numpy() if pred.boxes is not None else np.zeros((0,))
        confs = pred.boxes.conf.detach().cpu().numpy() if pred.boxes is not None else np.zeros((0,))

        per_person = associate_ppe_to_persons(boxes, clss, confs, w, h)
        annotated = draw_person_overlays(frame.copy(), per_person)

        # output writer init
        if writer is None:
            writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), fps_in, (annotated.shape[1], annotated.shape[0]))

        # FPS overlay
        end = time.time()
        frame_times.append(end-start)
        fps = 1.0 / max(1e-6, (end-start))
        cv2.putText(annotated, f"FPS: {fps:.2f}", (16, 32), cv2.FONT_HERSHEY_SIMPLEX, 1.0, WHITE, 2)
        writer.write(annotated)

        # Check violations per person and log events (with pre/post clip)
        for info in per_person:
            if info['violations']:
                # prepare post frames collection for this person if not already active
                remain = post_collect.get(info['pid'], 0)
                if remain <= 0:
                    # trigger save: take ring frames + we will collect POST_N frames now
                    post_collect[info['pid']] = int(POST_SEC * fps_in)

                    # Copy last pre frames now; post frames to be collected in next iterations
                    pre_copy = rb.dump()[-int(PRE_SEC*fps_in):] if PRE_SEC > 0 else []
                    # to mark pending save, stash pre frames in dict
                    post_collect[f"pre_{info['pid']}"] = pre_copy
                else:
                    # already in post-collection window
                    pass

        # Decrement post windows and commit evidence when they hit 0
        finish_list = []
        for k, v in post_collect.items():
            if isinstance(k, int):
                if v > 0:
                    post_collect[k] = v - 1
                    if v - 1 == 0:
                        finish_list.append(k)

        for pid in finish_list:
            # gather post frames from ring buffer (latest window)
            post_frames = []
            # we don't have exact separate storage, so we pull from last seconds as approximation
            post_frames = rb.dump()[-int(POST_SEC*fps_in):] if POST_SEC > 0 else []
            pre_frames  = post_collect.get(f"pre_{pid}", [])

            # find current person's most recent state & confidence
            # (approximate: reuse last computed per_person for this frame)
            person = None
            for info in per_person:
                if info['pid'] == pid:
                    person = info
                    break
            if person is None:
                # fallback: skip (unlikely)
                continue

            ev.save_event(
                video_name=video_name,
                frame_idx=frame_idx,
                person_id=pid,
                vtypes=person['violations'],
                conf=person['score'],
                ring_frames=pre_frames,
                post_frames=post_frames
            )
            # cleanup
            post_collect.pop(pid, None)
            post_collect.pop(f"pre_{pid}", None)

        frame_idx += 1

    # wrap up
    cap.release()
    if writer is not None:
        writer.release()

    total = time.time() - t0_all
    avg = float(np.mean(frame_times)) if frame_times else 0.0
    fps_avg = 1.0 / max(1e-6, avg)
    print(f"✅ Done: {video_name} | Avg FPS: {fps_avg:.2f} | Total time: {total:.1f}s")

    # Save performance report
    perf_row = {
        "video": video_name,
        "avg_fps": round(fps_avg, 2),
        "avg_inference_time_ms": round(avg*1000, 1),
        "frames_processed": frame_idx,
        "device": DEVICE,
        "imgsz": IMG_SIZE,
        "frame_resize": f"{FRAME_RESIZE}" if FRAME_RESIZE else "native",
        "frame_skip": FRAME_SKIP,
        "fp16": USE_FP16
    }
    perf_csv = os.path.join(REPORT_DIR, "performance_summary.csv")
    if not os.path.exists(perf_csv):
        pd.DataFrame(columns=list(perf_row.keys())).to_csv(perf_csv, index=False)
    pd.DataFrame([perf_row]).to_csv(perf_csv, mode="a", header=False, index=False)

In [11]:
# ---- PATCH: helpers required by process_video() ----
import cv2, os, time, csv, json
import numpy as np
from collections import deque
from datetime import datetime

def ts_now():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

class RingBuffer:
    """Fixed-length buffer that stores recent frames (BGR)."""
    def __init__(self, capacity: int):
        self.buf = deque(maxlen=max(1, capacity))
    def push(self, frame, _ts=None):
        # store only the frame; timestamp optional
        self.buf.append(frame.copy())
    def dump(self):
        # return a shallow copy list of frames
        return list(self.buf)

def _iou_xyxy(a, b):
    xA = max(a[0], b[0]); yA = max(a[1], b[1])
    xB = min(a[2], b[2]); yB = min(a[3], b[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    if inter <= 0: return 0.0
    areaA = max(0, a[2]-a[0]) * max(0, a[3]-a[1])
    areaB = max(0, b[2]-b[0]) * max(0, b[3]-b[1])
    union = areaA + areaB - inter + 1e-6
    return inter / union

class SimpleTracker:
    """
    Minimal IoU-based tracker.
    update(input_boxes) -> list of track_ids aligned to input_boxes.
    Keeps tracks alive for `max_miss` frames.
    """
    def __init__(self, iou_threshold=0.45, max_miss=30):
        self.iou_thr = iou_threshold
        self.max_miss = max_miss
        self.next_id = 1
        # id -> dict(bbox, miss)
        self.tracks = {}

    def _match(self, inputs):
        # greedy match tracks to inputs
        matches = {}          # track_id -> input_index
        used_inputs = set()
        for tid, t in list(self.tracks.items()):
            best_iou, best_j = 0.0, -1
            for j, box in enumerate(inputs):
                if j in used_inputs:
                    continue
                iou = _iou_xyxy(t["bbox"], box)
                if iou > best_iou:
                    best_iou, best_j = iou, j
            if best_j >= 0 and best_iou >= self.iou_thr:
                matches[tid] = best_j
                used_inputs.add(best_j)
            else:
                # not matched this frame
                t["miss"] += 1
        return matches, used_inputs

    def update(self, input_boxes):
        # ensure lists
        inputs = [list(map(int, b)) for b in input_boxes] if input_boxes else []
        # match existing tracks
        matches, used_inputs = self._match(inputs)

        # update matched tracks
        for tid, j in matches.items():
            self.tracks[tid]["bbox"] = inputs[j]
            self.tracks[tid]["miss"] = 0

        # create new tracks for unmatched inputs
        for j, box in enumerate(inputs):
            if j not in used_inputs:
                self.tracks[self.next_id] = {"bbox": box, "miss": 0}
                matches[self.next_id] = j
                self.next_id += 1

        # drop stale tracks
        for tid in list(self.tracks.keys()):
            if self.tracks[tid]["miss"] > self.max_miss:
                self.tracks.pop(tid, None)

        # build ids aligned to input order
        id_by_input = [-1] * len(inputs)
        for tid, j in matches.items():
            id_by_input[j] = tid
        return id_by_input

class EvidenceLogger:
    """
    Saves snapshot + concatenated clip (pre + post frames) and logs a CSV row.
    Root is typically your OUTPUT_DIR; it will write under:
      root/violations/YYYYMMDD/worker_<id>/<ViolationType>/
    """
    def __init__(self, root_dir, fps, pre_sec=5, post_sec=5):
        self.root = root_dir
        self.fps = fps
        self.pre_sec = pre_sec
        self.post_sec = post_sec
        os.makedirs(self.root, exist_ok=True)
        self.csv_path = os.path.join(self.root, "violation_metadata.csv")

    def _ensure(self, path):
        os.makedirs(path, exist_ok=True)

    def _write_clip(self, out_path, frames):
        if not frames:
            return
        h, w = frames[0].shape[:2]
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        vw = cv2.VideoWriter(out_path, fourcc, self.fps, (w, h))
        for f in frames:
            if f.shape[0] != h or f.shape[1] != w:
                f = cv2.resize(f, (w, h))
            vw.write(f)
        vw.release()

    def save_event(self, video_name, frame_idx, person_id, vtypes, conf, ring_frames, post_frames):
        # vtypes is a list; use the first for folder naming
        vtype = vtypes[0] if vtypes else "Violation"
        date_folder = datetime.now().strftime("%Y%m%d")
        base = os.path.join(self.root, "violations", date_folder, f"worker_{int(person_id)}", vtype)
        self._ensure(base)

        stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        snap_path = os.path.join(base, f"{stamp}_{vtype}_frame{frame_idx}.jpg")
        clip_path = os.path.join(base, f"{stamp}_{vtype}.mp4")

        # snapshot = last available frame from post or ring
        if post_frames:
            cv2.imwrite(snap_path, post_frames[min(len(post_frames)-1, 0)])
        elif ring_frames:
            cv2.imwrite(snap_path, ring_frames[-1])

        # full clip = ring (pre) + post
        frames = (ring_frames or []) + (post_frames or [])
        self._write_clip(clip_path, frames)

        row = {
            "timestamp": ts_now(),
            "video": video_name,
            "person_id": int(person_id),
            "violation_type": vtype,
            "confidence": conf if conf is not None else "",
            "frame_index": frame_idx,
            "snapshot_path": snap_path,
            "clip_path": clip_path
        }
        # append CSV
        file_exists = os.path.exists(self.csv_path)
        with open(self.csv_path, "a", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=list(row.keys()))
            if not file_exists:
                writer.writeheader()
            writer.writerow(row)
# ---- END PATCH ----


In [12]:

# ================================
# 12) Processing Pipeline WITH 3-FRAME CONFIRMATION
#     - adds SimpleTracker for stable IDs
#     - maintains per-ID, per-violation deque(maxlen=3)
#     - only fires evidence after 3 consecutive True
# ================================
from collections import defaultdict, deque

def process_video(path):
    video_name = os.path.basename(path)
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        print("❌ Cannot open:", path); return

    fps_in = cap.get(cv2.CAP_PROP_FPS) or 25.0
    writer = None
    out_path = os.path.join(OUTPUT_DIR, f"{os.path.splitext(video_name)[0]}_annotated.mp4")

    # buffers & loggers
    rb = RingBuffer(capacity=int(PRE_SEC * fps_in))
    ev = EvidenceLogger(OUTPUT_DIR, fps=fps_in, pre_sec=PRE_SEC, post_sec=POST_SEC)

    # NEW: tracker & history for 3-frame confirmation
    tracker = SimpleTracker(iou_threshold=0.45, max_miss=30)
    # history[tid][vtype] -> deque of bool, last 3 frames
    history = defaultdict(lambda: defaultdict(lambda: deque(maxlen=3)))
    # post_collect[(tid, vtype)] -> remaining frames; "pre_(tid,vtype)" holds pre-frames
    post_collect = {}

    frame_idx = 0
    t0_all = time.time()
    frame_times = []

    while True:
        ok, frame = cap.read()
        if not ok:
            break

        if FRAME_RESIZE is not None:
            frame = cv2.resize(frame, FRAME_RESIZE)

        rb.push(frame, ts_now())

        if (frame_idx % FRAME_SKIP) != 0:
            if writer is not None:
                writer.write(frame)
            frame_idx += 1
            continue

        h, w = frame.shape[:2]
        start = time.time()
        preds = model.predict(
            frame,
            conf=CONF_THRES,
            iou=IOU_THRES,
            imgsz=IMG_SIZE,
            device=DEVICE,
            half=USE_FP16,
            verbose=False
        )
        pred = preds[0]
        boxes = pred.boxes.xyxy.detach().cpu().numpy() if pred.boxes is not None else np.zeros((0,4))
        clss  = pred.boxes.cls.detach().cpu().numpy() if pred.boxes is not None else np.zeros((0,))
        confs = pred.boxes.conf.detach().cpu().numpy() if pred.boxes is not None else np.zeros((0,))

        per_person = associate_ppe_to_persons(boxes, clss, confs, w, h)

        # -------- tracker IDs for persons
        person_boxes = [pp['person_box'] for pp in per_person]
        tids = tracker.update(person_boxes)
        for i, pp in enumerate(per_person):
            pp['tid'] = tids[i] if i < len(tids) else None

        annotated = draw_person_overlays(frame.copy(), per_person)

        # init writer
        if writer is None:
            writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), fps_in, (annotated.shape[1], annotated.shape[0]))

        end = time.time()
        frame_times.append(end-start)
        fps = 1.0 / max(1e-6, (end-start))
        cv2.putText(annotated, f"FPS: {fps:.2f}", (16, 32), cv2.FONT_HERSHEY_SIMPLEX, 1.0, WHITE, 2)
        writer.write(annotated)

        # -------- 3-frame confirmation logic per (tid, vtype)
        V_TYPES = ["No Helmet", "No Vest", "No Mask"]
        for info in per_person:
            tid = info.get('tid')
            if tid is None:
                continue

            for vt in V_TYPES:
                is_now = (vt in info['violations'])
                dq = history[tid][vt]
                dq.append(bool(is_now))

                # Confirm only when last 3 are True
                if len(dq) == dq.maxlen and all(dq):
                    key = (tid, vt)
                    if key not in post_collect or post_collect[key] <= 0:
                        # start post-collection window and store pre-frames snapshot
                        post_collect[key] = int(POST_SEC * fps_in)
                        post_collect[("pre", tid, vt)] = rb.dump()[-int(PRE_SEC*fps_in):] if PRE_SEC > 0 else []

        # -------- decrement post timers, and when 0 -> save evidence
        finished = []
        for k in list(post_collect.keys()):
            if isinstance(k, tuple) and len(k) == 2:
                # (tid, vtype)
                remain = post_collect[k]
                if remain > 0:
                    post_collect[k] = remain - 1
                    if remain - 1 == 0:
                        finished.append(k)

        for (tid, vt) in finished:
            pre_frames = post_collect.get(("pre", tid, vt), [])
            post_frames = rb.dump()[-int(POST_SEC*fps_in):] if POST_SEC > 0 else []

            # get the latest person state for this tid
            person = None
            for info in per_person:
                if info.get('tid') == tid:
                    person = info
                    break
            if person is None:
                # cannot resolve this frame; skip save
                post_collect.pop((tid, vt), None)
                post_collect.pop(("pre", tid, vt), None)
                continue

            # Only save event for that single vtype (confirmed)
            ev.save_event(
                video_name=video_name,
                frame_idx=frame_idx,
                person_id=int(tid),
                vtypes=[vt],
                conf=person['score'],
                ring_frames=pre_frames,
                post_frames=post_frames
            )

            post_collect.pop((tid, vt), None)
            post_collect.pop(("pre", tid, vt), None)

        frame_idx += 1

    # wrap up
    cap.release()
    if writer is not None:
        writer.release()

    total = time.time() - t0_all
    avg = float(np.mean(frame_times)) if frame_times else 0.0
    fps_avg = 1.0 / max(1e-6, avg)
    print(f"✅ Done: {video_name} | Avg FPS: {fps_avg:.2f} | Total time: {total:.1f}s")

    # performance row
    perf_row = {
        "video": video_name,
        "avg_fps": round(fps_avg, 2),
        "avg_inference_time_ms": round(avg*1000, 1),
        "frames_processed": frame_idx,
        "device": DEVICE,
        "imgsz": IMG_SIZE,
        "frame_resize": f"{FRAME_RESIZE}" if FRAME_RESIZE else "native",
        "frame_skip": FRAME_SKIP,
        "fp16": USE_FP16,
        "confirm_frames": 3
    }
    perf_csv = os.path.join(REPORT_DIR, "performance_summary.csv")
    if not os.path.exists(perf_csv):
        pd.DataFrame(columns=list(perf_row.keys())).to_csv(perf_csv, index=False)
    pd.DataFrame([perf_row]).to_csv(perf_csv, mode="a", header=False, index=False)


# ================================
# Run on all videos (same as before)
# ================================
video_files = [os.path.join(VIDEO_DIR, f) for f in os.listdir(VIDEO_DIR)
               if f.lower().endswith((".mp4",".avi",".mov",".mkv"))]
if not video_files:
    print(f"⚠️ No videos found in {VIDEO_DIR}. Please add files and re-run.")
else:
    print("🎬 Found videos:", [os.path.basename(v) for v in video_files])
    for vp in video_files:
        process_video(vp)

print("\n📁 Evidence root:", OUTPUT_DIR)
print("📝 Metadata CSV:", os.path.join(REPORT_DIR, "violation_metadata.csv"))
print("📈 Perf CSV    :", os.path.join(REPORT_DIR, "performance_summary.csv"))


🎬 Found videos: ['demo_1.mp4', 'test.mp4']
✅ Done: demo_1.mp4 | Avg FPS: 48.43 | Total time: 97.6s
✅ Done: test.mp4 | Avg FPS: 45.52 | Total time: 35.6s

📁 Evidence root: /content/drive/MyDrive/SafetyEye/violations
📝 Metadata CSV: /content/drive/MyDrive/SafetyEye/violations_report/violation_metadata.csv
📈 Perf CSV    : /content/drive/MyDrive/SafetyEye/violations_report/performance_summary.csv
